# Phase 12 Wave 1: GraphRAG + Structured Debate

**Project:** HiFi — High-Fidelity Financial Intelligence  
**Phase:** 12 — GraphRAG + Structured Debate  
**Wave:** 1 (P12-E0-T1, P12-E1-T1, P12-E1-T2, P12-E3-T1)  

---

## How to use this notebook

This notebook is a **live didactic layer**. Every code cell imports from and calls into
existing `src/hifi/` modules or loads outputs from `scripts/`. It does not reimplement
any process already present in the codebase.

**No LLM calls. No servers required. All cells are runnable offline.**

One artifact must be generated before Section 2 (compliance analysis):

```bash
# From repo root — generates data/training/technical_compliance_v2.jsonl
# Requires data/market/*.parquet (run: make acquire-data-phase10 first)
uv run python scripts/generate_compliance_examples.py
```

All other cells (Sections 3–5) run without any prerequisite — they build artifacts live.

### What Wave 1 delivered

| Ticket | Deliverable | Module |
|---|---|---|
| P12-E0-T1 | `technical_compliance_v2.jsonl` (>=200 examples) | `scripts/generate_compliance_examples.py` |
| P12-E1-T1 | `FinancialGraph` class | `src/hifi/knowledge/graph_store.py` |
| P12-E1-T2 | `build_financial_graph()` + seeds | `src/hifi/knowledge/graph_construction.py` |
| P12-E3-T1 | `DebateTurn`, `DebateTranscript`, helpers | `src/hifi/collective/debate.py` |


In [ ]:
"""Setup: path resolution and imports from existing HiFi modules. Run this cell first."""
import json
import sys
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

warnings.filterwarnings('ignore')

# Resolve repo root regardless of working directory
_nb = Path('.').resolve()
ROOT = _nb.parent if (_nb.parent / 'src').exists() else _nb
sys.path.insert(0, str(ROOT / 'src'))

# ---------------------------------------------------------------------------
# HiFi module imports — source of truth lives in src/hifi/
# This notebook CALLS the existing code; it does not reimplement it.
# ---------------------------------------------------------------------------
from hifi.knowledge.graph_store import FinancialGraph
from hifi.knowledge.graph_construction import (
    DEFAULT_COMPETITORS,
    DEFAULT_MACRO_SENSITIVITY,
    _TICKER_FALLBACK,  # internal constant — deterministic offline metadata
    build_financial_graph,
)
from hifi.collective.debate import (
    DebateTranscript,
    DebateTurn,
    compute_vote_delta,
    identify_minority,
)
from hifi.agents.schemas import AgentSignal

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

DATA     = ROOT / 'data'
TRAINING = DATA / 'training'
KG_DIR   = DATA / 'knowledge_graph'
V2_JSONL = TRAINING / 'technical_compliance_v2.jsonl'
KG_JSON  = KG_DIR / 'financial_graph.json'

def _status(p: Path) -> str:
    return 'OK' if p.exists() else 'MISSING'

print(f'Repo root        : {ROOT}')
print(f'v2 JSONL         : {_status(V2_JSONL)}')
print(f'KG JSON (output) : {_status(KG_JSON)}')
print()
if not V2_JSONL.exists():
    print('NOTICE: Run the compliance script before Section 2:')
    print('  uv run python scripts/generate_compliance_examples.py')


---
## 1  Phase 12 Scientific Questions

### 1.1  Two independent improvement mechanisms

Phase 11 established that LoRA fine-tuning converges reliably on financial reasoning
tasks. Phase 12 investigates two complementary mechanisms that operate at different
levels of the system:

| Mechanism | Level | Open Question | David Section |
|---|---|---|---|
| **GraphRAG** | Knowledge retrieval | OQ-K02: does graph-guided retrieval improve P@k >= 5%? | SS11.3 |
| **Structured Debate** | Collective process | OQ-D01: does debate increase herding (kappa)? | SS12.2.4 |

These are scientifically independent: GraphRAG affects what *information* agents see,
debate affects how agents *reason collectively* from that information.

### 1.2  Ensemble error decomposition (review from Phase 11)

For an ensemble of $M$ agents with individual error variance $v$ and pairwise
correlation $\rho$ (the diversity metric):

$$E_{\text{ens}} = b^2 + \rho v + \frac{(1-\rho)v}{M}$$

**GraphRAG** targets $b^2$ (shared bias): better retrieval reduces the systematic
error all agents make when they retrieve irrelevant context.

**Structured Debate** changes $\rho$: if it reduces diversity (herding), $E_{\text{ens}}$
rises. If it corrects errors without synchronising agents, $\rho$ may fall.
Sunstein (2006) shows deliberation often causes *group polarisation* — the group
moves to a more extreme position than the average individual. Phase 12 measures this directly.

### 1.3  The 2x2 factorial (DJ-067)

| | No debate | With debate |
|---|---|---|
| **Base models** | A | B |
| **Fine-tuned** | C | D |

Interaction effect: $(D - B) - (C - A)$.  
A positive interaction means fine-tuning and debate are *synergistic* — specialised
agents with heterogeneous priors are better positioned to produce useful debate.
A negative interaction means debate erases the diversity advantage of fine-tuning (herding).

### 1.4  Open questions for Phase 12

| ID | Question | Phase |
|---|---|---|
| OQ-K02 | Does GraphRAG improve retrieval P@k >= 5% vs flat RAG? | 12 |
| OQ-M02 | Does multi-date evaluation confirm diversity preservation? | 12 |
| OQ-D01 | Does debate increase herding (kappa)? | 12 |
| OQ-D02 | Is the 2x2 interaction effect positive? | 12 |


In [ ]:
"""Theory: ensemble error vs diversity, and what debate/GraphRAG change."""
rho_arr = np.linspace(0, 1, 300)
M, v, b2 = 5, 1.0, 0.10
E_ens    = b2 + rho_arr * v + (1 - rho_arr) * v / M
E_single = b2 + v

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ---- Left: ensemble error curve ----
ax = axes[0]
ax.plot(rho_arr, E_ens, lw=2, color='steelblue', label=f'Ensemble (M={M})')
ax.axhline(E_single, lw=1.4, ls='--', color='firebrick', label='Single agent')
ax.fill_between(rho_arr, E_ens, E_single, where=E_ens < E_single,
                alpha=0.12, color='steelblue', label='Ensemble benefit zone')
# Where GraphRAG helps (lower b^2) vs debate risk (higher rho)
ax.annotate('GraphRAG\nlowers b\u00b2', xy=(0.3, 0.55), fontsize=8, color='seagreen',
            xytext=(0.05, 0.75), arrowprops={'arrowstyle': '->', 'color': 'seagreen'})
ax.annotate('Debate herding\nraises \u03c1', xy=(0.8, 0.96), fontsize=8, color='orange',
            xytext=(0.5, 1.10), arrowprops={'arrowstyle': '->', 'color': 'orange'})
ax.set(xlabel='Pairwise correlation \u03c1', ylabel='Expected ensemble error',
       title='Figure 1a: Ensemble error vs diversity')
ax.legend(fontsize=8)

# ---- Middle: 2x2 factorial conditions ----
ax2 = axes[1]
labels = ['A\nBase/No-debate', 'B\nBase/Debate', 'C\nFT/No-debate', 'D\nFT/Debate']
# Hypothetical expected error under each condition (illustrative)
errors = [1.10, 1.05, 0.95, 0.85]
colors = ['steelblue', 'cornflowerblue', 'seagreen', 'darkgreen']
bars = ax2.bar(labels, errors, color=colors, alpha=0.82, width=0.55)
for bar, e in zip(bars, errors):
    ax2.text(bar.get_x() + bar.get_width() / 2, e + 0.01, f'{e}', ha='center', fontsize=8)
ax2.axhline(E_single, lw=1.2, ls='--', color='firebrick', label='Single agent baseline')
ax2.set(ylabel='Expected error (illustrative)', ylim=(0.5, 1.35),
        title='Figure 1b: 2x2 factorial conditions (DJ-067)')
ax2.legend(fontsize=8)
# Annotate interaction
ax2.annotate('', xy=(3, 0.85), xytext=(2, 0.95),
             arrowprops={'arrowstyle': '<->', 'color': 'purple', 'lw': 1.5})
ax2.text(2.5, 0.87, 'D-C', ha='center', fontsize=8, color='purple')
ax2.annotate('', xy=(1, 1.05), xytext=(0, 1.10),
             arrowprops={'arrowstyle': '<->', 'color': 'gray', 'lw': 1.5})
ax2.text(0.5, 1.07, 'B-A', ha='center', fontsize=8, color='gray')

# ---- Right: what herding looks like in vote space ----
ax3 = axes[2]
# Pre-debate: 3 Buy, 1 Sell, 1 Hold (diverse)
# Post-debate herding: 4 Buy, 0 Sell, 1 Hold
# Post-debate diverging: 2 Buy, 2 Sell, 1 Hold
scenarios = ['Pre-debate', 'Post-debate\n(herded)', 'Post-debate\n(diverged)']
buy_  = [3, 4, 2]
hold_ = [1, 1, 1]
sell_ = [1, 0, 2]
x = np.arange(len(scenarios))
w = 0.28
ax3.bar(x - w, buy_,  w, color='seagreen',   alpha=0.82, label='Buy')
ax3.bar(x,     hold_, w, color='goldenrod',  alpha=0.82, label='Hold')
ax3.bar(x + w, sell_, w, color='firebrick',  alpha=0.82, label='Sell')
ax3.set(xticks=x, xticklabels=scenarios, ylabel='Vote count',
        ylim=(0, 5.5), title='Figure 1c: Herding vs divergence in vote space')
ax3.legend(fontsize=8)
# Labels
for i, (b, h, s) in enumerate(zip(buy_, hold_, sell_)):
    ax3.text(i - w, b + 0.08, str(b), ha='center', fontsize=8)
    ax3.text(i,     h + 0.08, str(h), ha='center', fontsize=8)
    ax3.text(i + w, s + 0.08, str(s), ha='center', fontsize=8)

plt.suptitle('Figure 1: Phase 12 theoretical framework', fontsize=11)
plt.tight_layout()
plt.show()


---
## 2  E0: Technical_v1 Compliance Fix (DJ-061)

### 2.1  Root cause: the 0.19% compliance ratio

Phase 11 trained `technical_v1` on 26,433 examples where ~50 were compliance examples
(verified, HR=0, GR=1 outputs from Phases 3/5). The model weight update was dominated
by the 26,383 domain examples. The compliance objective — teaching the correct JSON
output schema — was drowned out.

Result: `technical_v1` GR collapsed from 1.000 (base) to 0.000 (fine-tuned) on the
evaluation date. The fine-tuned model produces syntactically malformed JSON.

**Fix (DJ-061):** Augment compliance examples from ~50 to >= 200 (0.75% ratio).
The augmentation strategy:
1. Extract verified examples from Phase 4 and Phase 9 fixtures (~6 total)
2. Generate synthetic format-compliant examples from OHLCV data at quarterly intervals
   using the existing `format_as_jsonl()` + `generate_max_return_labels()` functions
3. Target: 200 total (194 synthetic + 6 extracted)

### 2.2  The script that generated this artifact

```bash
# scripts/generate_compliance_examples.py  (modified for Phase 12)
#
# Key additions vs Phase 11 version:
#   - _generate_synthetic_technical_compliance(): discovers tickers from data/market/*.parquet,
#     calls generate_max_return_labels() + format_as_jsonl() at quarterly sampling (iloc[::63])
#   - Extended main(): extracts Phase 9 fixture, deduplicates, generates v2 output
#
uv run python scripts/generate_compliance_examples.py
# Output: data/training/technical_compliance_v2.jsonl
```

### 2.3  Why synthetic examples are safe

The synthetic examples use **real OHLCV indicators** (RSI, MACD, ATR, SMA via
`format_as_jsonl()`) with **oracle labels** (60-day forward return via
`generate_max_return_labels()`). The same pipeline was proven correct in Phase 11
for the 26,433 domain examples. We are reusing the same `training_data.py` functions —
only the sampling granularity changes (quarterly instead of daily).


In [ ]:
"""Load technical_compliance_v2.jsonl and inspect its structure."""
if not V2_JSONL.exists():
    print('SKIP: technical_compliance_v2.jsonl not found.')
    print('Run: uv run python scripts/generate_compliance_examples.py')
    examples_v2 = []
else:
    examples_v2 = [
        json.loads(ln)
        for ln in V2_JSONL.read_text().splitlines()
        if ln.strip()
    ]
    print(f'Total examples in v2 : {len(examples_v2)}')
    print(f'Target (DJ-061)       : >= 200')
    print(f'Status                : {"PASS" if len(examples_v2) >= 200 else "FAIL"}')
    print()

    # Inspect first example
    ex0 = examples_v2[0]
    print('First example structure:')
    for msg in ex0['messages']:
        role = msg['role'].upper()
        body = msg['content']
        preview = body[:200] + ' ...[truncated]' if len(body) > 200 else body
        print(f'  [{role}] {preview}')
        print()


In [ ]:
"""Analyse v2 JSONL: decision distribution, compliance ratio fix visualisation."""
if not examples_v2:
    print('SKIP: no data. Run generate_compliance_examples.py first.')
else:
    from collections import Counter

    decisions = []
    for ex in examples_v2:
        asst = next((m['content'] for m in ex['messages'] if m['role'] == 'assistant'), '{}')
        try:
            decisions.append(json.loads(asst).get('decision', '?'))
        except Exception:
            decisions.append('?')

    dc = Counter(decisions)
    COLORS = {'Buy': 'seagreen', 'Hold': 'goldenrod', 'Sell': 'firebrick', '?': 'gray'}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Plot A: compliance ratio before vs after
    ax = axes[0]
    domain_n  = 26_433
    v1_comp   = 50
    v2_comp   = len(examples_v2)
    v1_total  = domain_n + v1_comp
    v2_total  = domain_n + v2_comp
    v1_ratio  = v1_comp / v1_total * 100
    v2_ratio  = v2_comp / v2_total * 100
    bars = ax.bar(['Phase 11\ntechnical_v1', 'Phase 12\ntechnical_v2'],
                  [v1_ratio, v2_ratio],
                  color=['firebrick', 'seagreen'], alpha=0.82, width=0.45)
    for bar, r in zip(bars, [v1_ratio, v2_ratio]):
        ax.text(bar.get_x() + bar.get_width() / 2, r + 0.02,
                f'{r:.2f}%', ha='center', fontsize=9)
    ax.set(ylabel='Compliance example ratio (%)',
           title='Figure 2a: Compliance ratio fix (DJ-061)',
           ylim=(0, max(v1_ratio, v2_ratio) * 1.6))
    ax.text(0, v1_ratio + 0.09, f'N={v1_comp}/{v1_total:,}', ha='center', fontsize=8, color='gray')
    ax.text(1, v2_ratio + 0.09, f'N={v2_comp}/{v2_total:,}', ha='center', fontsize=8, color='gray')

    # Plot B: decision distribution in v2
    ax2 = axes[1]
    labels_pie   = [k for k in ['Buy', 'Hold', 'Sell'] if k in dc]
    sizes_pie    = [dc[k] for k in labels_pie]
    colors_pie   = [COLORS[k] for k in labels_pie]
    ax2.pie(sizes_pie, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'lw': 1.2})
    ax2.set_title(f'Figure 2b: Decision distribution\nin v2 (N={len(examples_v2)})')

    # Plot C: composition — extracted vs synthetic
    ax3 = axes[2]
    # The script extracts 6 verified examples and fills the rest synthetically
    n_extracted  = min(6, len(examples_v2))  # Phase 4 (3) + Phase 9 (3)
    n_synthetic  = len(examples_v2) - n_extracted
    ax3.barh(['Extracted\n(Phase 4+9\nverified)', 'Synthetic\n(OHLCV quarterly\n+ oracle labels)'],
             [n_extracted, n_synthetic],
             color=['steelblue', 'cornflowerblue'], alpha=0.82)
    for i, n in enumerate([n_extracted, n_synthetic]):
        ax3.text(n + 1, i, str(n), va='center', fontsize=9)
    ax3.set(xlabel='Examples', title='Figure 2c: Example composition')
    ax3.axvline(200, color='orange', lw=1.2, ls='--', label='Target=200')
    ax3.legend(fontsize=8)

    plt.suptitle('Figure 2: P12-E0-T1 — technical_compliance_v2.jsonl analysis', fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f'Extracted (verified fixtures): {n_extracted}')
    print(f'Synthetic (OHLCV quarterly):  {n_synthetic}')
    print(f'Total:                        {len(examples_v2)}')
    print(f'Compliance ratio:             {v2_ratio:.2f}% (was {v1_ratio:.2f}%)')
    print(f'Root cause fix:               4x increase in compliance signal')


---
## 3  E1: FinancialGraph — Building the Knowledge Graph (P12-E1-T1 + E1-T2)

### 3.1  Architecture decision (DJ-062, DJ-063)

**DJ-062:** GraphRAG uses NetworkX + LanceDB. No new venv. NetworkX is the in-process
graph engine; LanceDB (already present) handles dense retrieval. The graph is a
*pre-filter* stage: it expands a query ticker to related tickers, which become
a filter applied to the vector store.

**DJ-063:** Tight scope — ~12 company nodes, 3 macro nodes, ~40 edges. The graph is
curated (not crawled from Wikipedia or yfinance at scale) because the Phase 12
evaluation universe is 3 tickers + their sector peers. Over-engineering the graph
at this stage would violate DJ-016 (do not add complexity without evidence).

### 3.2  Graph schema

| Node type | Identifier | Key attributes |
|---|---|---|
| Company | ticker (str) | name, sector, industry |
| Sector | name (str) | — |
| MacroFactor | name (str) | series_id (FRED series) |

| Edge type | Direction | Semantics |
|---|---|---|
| BELONGS_TO | Company → Sector | ticker is in this sector |
| COMPETES_WITH | Company ↔ Company | symmetric, stored as 2 directed edges |
| SENSITIVE_TO | Sector → MacroFactor | macro factor affects this sector |

### 3.3  Query expansion (DJ-064)

Given a query ticker, `expand_query_tickers(ticker, max_hops=2)` returns:
- hop=0: just the ticker
- hop=1: + direct competitors (COMPETES_WITH edges)
- hop=2: + sector peers (other companies in the same Sector node)

The expanded ticker set is passed as a filter to LanceDB dense search. Documents
that mention any of these tickers are retrieved — capturing context the user did
not explicitly ask for but that is structurally related.


In [ ]:
"""Build the financial graph live using build_financial_graph() from graph_construction.py.

We pass ticker_metadata=_TICKER_FALLBACK to avoid yfinance network calls.
_TICKER_FALLBACK is the hardcoded offline metadata for the 17 Phase 10 tickers.
In production (live runs), ticker_metadata=None triggers yfinance lookup with
fallback to the same constant when the network is unavailable.
"""
# Phase 12 evaluation universe + sector peers
PHASE12_TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'NVDA', 'AMZN', 'META',  # Technology
    'JPM', 'BAC', 'GS',                                  # Financial Services
    'XOM', 'CVX',                                         # Energy
]

# build_financial_graph() is defined in src/hifi/knowledge/graph_construction.py
# We call it directly — the notebook does not reimplement the construction logic.
g = build_financial_graph(
    tickers=PHASE12_TICKERS,
    competitor_seed=DEFAULT_COMPETITORS,
    macro_sensitivity=DEFAULT_MACRO_SENSITIVITY,
    ticker_metadata=_TICKER_FALLBACK,  # offline: skip yfinance
)

print(f'Graph built: {g.node_count()} nodes, {g.edge_count()} edges')
print()

# Break down by node type
nxg = g._g  # expose the NetworkX DiGraph for inspection
company_nodes  = [n for n, d in nxg.nodes(data=True) if d.get('node_type') == 'company']
sector_nodes   = [n for n, d in nxg.nodes(data=True) if d.get('node_type') == 'sector']
macro_nodes    = [n for n, d in nxg.nodes(data=True) if d.get('node_type') == 'macro_factor']

print(f'Company nodes  ({len(company_nodes)}): {sorted(company_nodes)}')
print(f'Sector nodes   ({len(sector_nodes)}):  {sorted(sector_nodes)}')
print(f'Macro nodes    ({len(macro_nodes)}):   {sorted(macro_nodes)}')

# Break down by edge type
belongs  = [(u, v) for u, v, d in nxg.edges(data=True) if d.get('edge_type') == 'BELONGS_TO']
competes = [(u, v) for u, v, d in nxg.edges(data=True) if d.get('edge_type') == 'COMPETES_WITH']
sensitiv = [(u, v) for u, v, d in nxg.edges(data=True) if d.get('edge_type') == 'SENSITIVE_TO']
print()
print(f'BELONGS_TO     ({len(belongs)}):  {len(belongs)} company -> sector')
print(f'COMPETES_WITH  ({len(competes)}): {len(competes)} directed (symmetric pairs)')
print(f'SENSITIVE_TO   ({len(sensitiv)}):  {len(sensitiv)} sector -> macro')


In [ ]:
"""Visualise the financial graph with NetworkX + matplotlib.

Node colours:
  steelblue   = Company
  goldenrod   = Sector
  firebrick   = MacroFactor

Edge styles:
  solid grey        = BELONGS_TO  (Company -> Sector)
  dashed steelblue  = COMPETES_WITH (symmetric)
  dashed orange     = SENSITIVE_TO  (Sector -> MacroFactor)
"""
NODE_COLORS = {'company': 'steelblue', 'sector': 'goldenrod', 'macro_factor': 'firebrick'}

node_colors = [NODE_COLORS.get(nxg.nodes[n].get('node_type'), 'gray') for n in nxg.nodes]
node_sizes  = [600 if nxg.nodes[n].get('node_type') == 'company'
               else 900 if nxg.nodes[n].get('node_type') == 'sector'
               else 800 for n in nxg.nodes]

# Separate edges by type for different styling
edges_bt  = [(u, v) for u, v, d in nxg.edges(data=True) if d.get('edge_type') == 'BELONGS_TO']
edges_cw  = [(u, v) for u, v, d in nxg.edges(data=True) if d.get('edge_type') == 'COMPETES_WITH']
edges_st  = [(u, v) for u, v, d in nxg.edges(data=True) if d.get('edge_type') == 'SENSITIVE_TO']

fig, ax = plt.subplots(figsize=(14, 10))

# Hierarchical-style layout: sectors and macros at top, companies below
pos = nx.spring_layout(nxg, seed=42, k=2.2)
# Manually fix macro and sector nodes higher for readability
for n in sector_nodes:
    pos[n] = (pos[n][0], pos[n][1] + 1.0)
for n in macro_nodes:
    pos[n] = (pos[n][0], pos[n][1] + 2.0)

nx.draw_networkx_nodes(nxg, pos, node_color=node_colors, node_size=node_sizes,
                       alpha=0.88, ax=ax)
nx.draw_networkx_labels(nxg, pos, font_size=7, font_color='white', ax=ax)

# Draw each edge type with distinct style
nx.draw_networkx_edges(nxg, pos, edgelist=edges_bt,
                       edge_color='gray',       width=1.2, alpha=0.6,
                       arrows=True, arrowsize=12, ax=ax)
nx.draw_networkx_edges(nxg, pos, edgelist=edges_cw,
                       edge_color='steelblue',  width=1.5, alpha=0.5,
                       style='dashed', arrows=False, ax=ax)
nx.draw_networkx_edges(nxg, pos, edgelist=edges_st,
                       edge_color='orange',     width=1.8, alpha=0.7,
                       arrows=True, arrowsize=14, ax=ax)

# Legend
legend_handles = [
    mpatches.Patch(color='steelblue',  label='Company node'),
    mpatches.Patch(color='goldenrod',  label='Sector node'),
    mpatches.Patch(color='firebrick',  label='MacroFactor node'),
    mpatches.Patch(color='gray',       label='BELONGS_TO edge'),
    mpatches.Patch(color='steelblue',  label='COMPETES_WITH edge (dashed)'),
    mpatches.Patch(color='orange',     label='SENSITIVE_TO edge'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=8)
ax.set_title(
    f'Figure 3: HiFi Financial Knowledge Graph\n'
    f'{g.node_count()} nodes | {g.edge_count()} directed edges | '
    f'built via build_financial_graph() (graph_construction.py)',
    fontsize=10,
)
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
"""Demonstrate expand_query_tickers() — the core of the GraphRAG retrieval pre-filter.

expand_query_tickers() is defined in src/hifi/knowledge/graph_store.py.
We call it directly on the graph we just built.
"""
print('Query expansion: expand_query_tickers(ticker, max_hops)')
print('=' * 60)

eval_tickers = ['AAPL', 'JPM', 'XOM']  # the 3 Phase 10/12 evaluation tickers

rows = []
for ticker in eval_tickers:
    hop0 = g.expand_query_tickers(ticker, max_hops=0)
    hop1 = g.expand_query_tickers(ticker, max_hops=1)
    hop2 = g.expand_query_tickers(ticker, max_hops=2)
    rows.append({'ticker': ticker, 'hop=0': hop0, 'hop=1': hop1, 'hop=2': hop2})
    print(f'{ticker}:')
    print(f'  hop=0 (ticker only)  : {hop0}')
    print(f'  hop=1 (+ competitors): {hop1}')
    print(f'  hop=2 (+ sector peers): {hop2}')
    print()

# Visualise expansion breadth
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(eval_tickers))
w = 0.28
for i, (hop, color) in enumerate([(0, 'lightgray'), (1, 'steelblue'), (2, 'seagreen')]):
    sizes = [len(g.expand_query_tickers(t, max_hops=hop)) for t in eval_tickers]
    bars  = ax.bar(x + (i - 1) * w, sizes, w, color=color, alpha=0.82, label=f'hop={hop}')
    for bar, s in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width() / 2, s + 0.05, str(s),
                ha='center', fontsize=8)

ax.set(xticks=x, xticklabels=eval_tickers,
       ylabel='Tickers in expanded set',
       title='Figure 4: GraphRAG query expansion breadth\n'
             '(larger set = wider context window for LanceDB retrieval)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Sector peers and macro factors:')
for ticker in eval_tickers:
    peers = g.get_sector_peers(ticker)
    macros = g.get_macro_factors(ticker)
    print(f'  {ticker}: sector_peers={peers}  macro_factors={macros}')


In [ ]:
"""Save the graph to JSON and load it back — verifying the roundtrip.

FinancialGraph.save() / .load() use networkx.node_link_data / node_link_graph.
The JSON file is the persistent artifact used by GraphRetriever in Wave 2.
"""
import tempfile

# Save to the canonical output location
KG_DIR.mkdir(parents=True, exist_ok=True)
g.save(KG_JSON)
print(f'Saved: {KG_JSON}')
file_size_kb = KG_JSON.stat().st_size / 1024
print(f'File size: {file_size_kb:.1f} KB')
print()

# Load back and verify identity
g_loaded = FinancialGraph.load(KG_JSON)
assert g_loaded.node_count() == g.node_count(), 'Node count mismatch after roundtrip'
assert g_loaded.edge_count() == g.edge_count(), 'Edge count mismatch after roundtrip'

# Verify expand_query_tickers is identical
for ticker in eval_tickers:
    original = g.expand_query_tickers(ticker, max_hops=2)
    loaded   = g_loaded.expand_query_tickers(ticker, max_hops=2)
    assert original == loaded, f'{ticker}: expansion mismatch {original} vs {loaded}'

print(f'Roundtrip OK: {g_loaded.node_count()} nodes, {g_loaded.edge_count()} edges')
print('expand_query_tickers identical for all 3 evaluation tickers.')
print()

# Show first few lines of the JSON for reference
data = json.loads(KG_JSON.read_text())
print(f'JSON keys: {list(data.keys())}')
print(f'First node: {data["nodes"][0]}')


### 3.4  Why the 2-hop BFS is the right design

The alternative to graph-guided expansion is a flat ticker filter: only retrieve
documents explicitly mentioning the query ticker. This misses two important signal classes:

1. **Competitor context:** When AAPL reports earnings, MSFT and GOOGL analyst
   commentary often revises sector-wide outlooks. A flat filter for 'AAPL' would
   miss documents that say 'Google's cloud growth suggests sector resilience'
   even though that document is directly relevant to an AAPL outlook.

2. **Macro sensitivity by sector:** Technology sector documents about Fed rate
   decisions (FFR) are relevant to all tech companies, not just the one explicitly
   mentioned. The SENSITIVE_TO edges encode this domain knowledge.

The 2-hop limit is a deliberate constraint. 3-hop expansion from AAPL in a global
financial graph would include thousands of companies (through institutional cross-holdings).
2 hops keeps the expanded set under 10 tickers — enough to capture the relevant
cluster without destroying retrieval precision (OQ-K02 asks for P@k >= 5% improvement,
not for recall at any precision).


---
## 4  E3: Structured Debate — Oxford Protocol (P12-E3-T1)

### 4.1  The Oxford 1-round protocol (DJ-065)

The Oxford debate format imposes structure on multi-agent deliberation.
Phase 12 implements one round:

```
Phase 1: Independent analysis     (existing run_ensemble() — no change)
Phase 2: identify_minority()       <-- Wave 1 (E3-T1)
Phase 3: Challenge turns           <-- Wave 2 (E3-T2)
Phase 4: Response turns            <-- Wave 2 (E3-T2)
Phase 5: Revision phase            <-- Wave 2 (E3-T2)
Phase 6: Final vote                (run_all_methods() on revised signals)
```

Wave 1 (this phase) implements the **schemas and classifiers**: `DebateTurn`,
`DebateTranscript`, `identify_minority()`, and `compute_vote_delta()`. The
orchestration logic (who calls whom, in what order) is Wave 2.

### 4.2  Why minority-vs-majority, not random pairs

Random agent pairing would not produce structured deliberation — it would produce
noise. The minority/majority structure ensures:
1. Minority agents challenge a *concrete position*, not an abstract one
2. Majority agents must *defend* their position, which exposes reasoning gaps
3. The direction of vote change (converged/diverged) is interpretable relative
   to the pre-debate majority, enabling the herding measurement

### 4.3  Herding measurement (compute_vote_delta)

- **converged:** more agents agree with the initial majority after debate
  (herding signal — debate suppressed minority views)
- **diverged:** fewer agents agree with the initial majority after debate
  (the minority successfully persuaded some majority agents)
- **unchanged:** no agent changed its vote

Tracking `vote_delta` across 10 dates × 3 tickers in the 2x2 factorial provides
the empirical herding distribution, which answers OQ-D01.


In [ ]:
"""Demonstrate identify_minority() on representative vote patterns.

identify_minority() is defined in src/hifi/collective/debate.py.
We call it directly with constructed AgentSignal objects.
"""
def _make_signal(agent_type: str, decision: str, confidence: float = 0.75) -> AgentSignal:
    """Helper to build an AgentSignal for the debate demo."""
    return AgentSignal(
        ticker='AAPL',
        as_of_date='2024-03-31',
        agent_type=agent_type,
        decision=decision,
        confidence=confidence,
        rationale=f'{agent_type} analysis suggests {decision}.',
        key_concern='Market uncertainty',
        model_id='qwen2.5-coder-32b-instruct-mlx@1234',
    )

SCENARIOS = {
    'Unanimous Buy (no debate)': [
        _make_signal('technical',    'Buy',  0.85),
        _make_signal('fundamental',  'Buy',  0.80),
        _make_signal('risk',         'Buy',  0.70),
        _make_signal('macro',        'Buy',  0.75),
        _make_signal('sentiment',    'Buy',  0.90),
    ],
    '4 Buy, 1 Sell (risk dissents)': [
        _make_signal('technical',    'Buy',  0.85),
        _make_signal('fundamental',  'Buy',  0.80),
        _make_signal('risk',         'Sell', 0.70),
        _make_signal('macro',        'Buy',  0.75),
        _make_signal('sentiment',    'Buy',  0.90),
    ],
    '3 Buy, 2 Sell (contested)': [
        _make_signal('technical',    'Buy',  0.85),
        _make_signal('fundamental',  'Sell', 0.80),
        _make_signal('risk',         'Sell', 0.70),
        _make_signal('macro',        'Buy',  0.65),
        _make_signal('sentiment',    'Buy',  0.90),
    ],
    '2 Buy, 2 Sell, 1 Hold (3-way tie->Hold)': [
        _make_signal('technical',    'Buy',  0.85),
        _make_signal('fundamental',  'Sell', 0.80),
        _make_signal('risk',         'Hold', 0.70),
        _make_signal('macro',        'Sell', 0.65),
        _make_signal('sentiment',    'Buy',  0.90),
    ],
}

print('identify_minority() results:')
print('=' * 72)
for desc, signals in SCENARIOS.items():
    minority, majority = identify_minority(signals)
    debate_needed = len(minority) > 0
    print(f'\nScenario: {desc}')
    print(f'  Majority decision : {majority}')
    print(f'  Minority agents   : {minority if minority else "(none — unanimous)"}')
    print(f'  Debate triggered  : {debate_needed}')


In [ ]:
"""Demonstrate compute_vote_delta() — measures herding direction after debate.

compute_vote_delta() is defined in src/hifi/collective/debate.py.
"""
# Baseline: 4 Buy, 1 Sell  -> majority=Buy, minority=[risk]
signals_initial = [
    _make_signal('technical',   'Buy',  0.85),
    _make_signal('fundamental', 'Buy',  0.80),
    _make_signal('risk',        'Sell', 0.70),
    _make_signal('macro',       'Buy',  0.75),
    _make_signal('sentiment',   'Buy',  0.90),
]

# Revision scenarios after debate
revision_scenarios = {
    'Converged (risk changes to Buy)': [
        _make_signal('technical',   'Buy',  0.87),
        _make_signal('fundamental', 'Buy',  0.82),
        _make_signal('risk',        'Buy',  0.65),   # changed Sell -> Buy
        _make_signal('macro',       'Buy',  0.75),
        _make_signal('sentiment',   'Buy',  0.90),
    ],
    'Diverged (fundamental changes to Sell)': [
        _make_signal('technical',   'Buy',  0.85),
        _make_signal('fundamental', 'Sell', 0.72),   # changed Buy -> Sell
        _make_signal('risk',        'Sell', 0.70),
        _make_signal('macro',       'Buy',  0.75),
        _make_signal('sentiment',   'Buy',  0.90),
    ],
    'Unchanged (no revision)': signals_initial,
}

print('compute_vote_delta() results:')
print('Initial vote: 4 Buy, 1 Sell  ->  majority=Buy')
print('=' * 60)
delta_results = {}
for desc, revised in revision_scenarios.items():
    delta, n_changed = compute_vote_delta(signals_initial, revised)
    delta_results[desc] = (delta, n_changed)
    print(f'\n  {desc}')
    print(f'    vote_delta        = {delta}')
    print(f'    n_changed_vote    = {n_changed}')

# Visualise
fig, ax = plt.subplots(figsize=(10, 4))
delta_colors = {'converged': 'firebrick', 'diverged': 'seagreen', 'unchanged': 'goldenrod'}
labels_v = list(revision_scenarios.keys())
deltas   = [delta_results[d][0] for d in labels_v]
changed  = [delta_results[d][1] for d in labels_v]
colors_v = [delta_colors[d] for d in deltas]

bars = ax.bar(range(len(labels_v)), changed, color=colors_v, alpha=0.82, width=0.5)
ax.set_xticks(range(len(labels_v)))
ax.set_xticklabels([f'{d}\n[{delta_results[d][0]}]' for d in labels_v],
                   fontsize=8, ha='center')
ax.set(ylabel='Agents that changed vote', ylim=(0, 3.5),
       title='Figure 5: compute_vote_delta() — herding direction measurement')
for bar, n in zip(bars, changed):
    ax.text(bar.get_x() + bar.get_width() / 2, n + 0.05, str(n), ha='center', fontsize=10)

legend_handles = [mpatches.Patch(color=v, label=k) for k, v in delta_colors.items()]
ax.legend(handles=legend_handles, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
"""Build a complete DebateTranscript and verify it round-trips to JSON.

DebateTurn and DebateTranscript are defined in src/hifi/collective/debate.py.
This cell constructs a minimal but valid transcript that represents one debate
for AAPL on 2024-03-31 where the risk agent (minority) challenges a Buy majority.
"""
# Initial signals: 4 Buy, 1 Sell (risk)
initial = [
    _make_signal('technical',   'Buy',  0.85),
    _make_signal('fundamental', 'Buy',  0.80),
    _make_signal('risk',        'Sell', 0.70),
    _make_signal('macro',       'Buy',  0.75),
    _make_signal('sentiment',   'Buy',  0.90),
]

minority, majority = identify_minority(initial)

# Challenge: risk agent argues against Buy
challenge = DebateTurn(
    agent_type='risk',
    phase='challenge',
    argument=(
        'The current Sharpe ratio is 0.31 over the trailing 252 days, below the '
        'Buy threshold of 0.5. Momentum is positive but risk-adjusted returns '
        'do not justify a Buy signal at this volatility level (ATR=3.2, VIX=18.4). '
        'I maintain Sell.'
    ),
    model_id='qwen2.5-coder-32b-instruct-mlx@1234',
)

# Response: technical agent defends Buy
response = DebateTurn(
    agent_type='technical',
    phase='response',
    argument=(
        'The RSI(14) is 58 (not overbought), MACD histogram positive for 8 consecutive '
        'sessions, and price has held above the 50-day SMA for 22 days. '
        'Risk metrics reflect recent macro volatility (VIX spike), not company-specific risk. '
        'The momentum signal outweighs the Sharpe concern on a 60-day horizon.'
    ),
    model_id='qwen2.5-coder-32b-instruct-mlx@1234',
)

# Revision: risk agent partially updates
revision_risk = DebateTurn(
    agent_type='risk',
    phase='revision',
    argument=(
        'The technical momentum evidence is credible. I upgrade from Sell to Hold. '
        'The Sharpe concern remains valid but the 60-day horizon justifies '
        'a more neutral stance given the sustained SMA support.'
    ),
    revised_decision='Hold',
    revised_confidence=0.55,
    model_id='qwen2.5-coder-32b-instruct-mlx@1234',
)

# Revised signals (risk changes Sell -> Hold)
revised = [
    _make_signal('technical',   'Buy',  0.85),
    _make_signal('fundamental', 'Buy',  0.80),
    AgentSignal(ticker='AAPL', as_of_date='2024-03-31', decision='Hold',
                confidence=0.55, rationale='Upgraded from Sell to Hold after debate.',
                key_concern='Risk-adjusted return below Buy threshold'),
    _make_signal('macro',       'Buy',  0.75),
    _make_signal('sentiment',   'Buy',  0.90),
]

delta, n_changed = compute_vote_delta(initial, revised)

# Assemble the full transcript
transcript = DebateTranscript(
    ticker='AAPL',
    as_of_date='2024-03-31',
    initial_signals=initial,
    minority_agents=minority,
    majority_decision=majority,
    challenge_turns=[challenge],
    response_turns=[response],
    revised_signals=revised,
    vote_delta=delta,
    n_agents_changed_vote=n_changed,
    debate_skipped=False,
)

# Verify JSON roundtrip (required for Dataset Family D storage)
as_json = transcript.model_dump_json(indent=2)
restored = DebateTranscript.model_validate_json(as_json)
assert restored.vote_delta == transcript.vote_delta
assert restored.n_agents_changed_vote == transcript.n_agents_changed_vote

print('DebateTranscript summary:')
print(f'  ticker            : {transcript.ticker}')
print(f'  as_of_date        : {transcript.as_of_date}')
print(f'  majority_decision : {transcript.majority_decision}')
print(f'  minority_agents   : {transcript.minority_agents}')
print(f'  challenge_turns   : {len(transcript.challenge_turns)}')
print(f'  response_turns    : {len(transcript.response_turns)}')
print(f'  vote_delta        : {transcript.vote_delta}')
print(f'  n_agents_changed  : {transcript.n_agents_changed_vote}')
print(f'  debate_skipped    : {transcript.debate_skipped}')
print()
print(f'JSON roundtrip: OK ({len(as_json):,} chars)')
print()
print('Challenge argument (truncated):')
print(f'  {challenge.argument[:120]}...')
print('Response argument (truncated):')
print(f'  {response.argument[:120]}...')
print(f'Revision: risk agent changed to {revision_risk.revised_decision} '
      f'(confidence={revision_risk.revised_confidence})')


---
## 5  Backward Compatibility: EnsembleOutput.debate_transcript

A key design constraint is that all existing code using `EnsembleOutput` must
continue to work without modification. The `debate_transcript` field is added
as `Optional[DebateTranscript] = None` to `collective/schemas.py`.

```python
# src/hifi/collective/schemas.py  (modified in Wave 1)
from hifi.collective.debate import DebateTranscript

class EnsembleOutput(BaseModel):
    ...existing fields...
    debate_transcript: DebateTranscript | None = None   # NEW — Phase 12
```

When `debate_transcript is None`, the output is a Phase 9/10 compatible
no-debate run. When `debate_transcript.debate_skipped is True`, the initial
vote was unanimous — no debate was needed. When populated, the full transcript
is available for analysis, storage as Dataset Family D, and the 2x2 factorial.


In [ ]:
"""Verify backward compatibility: EnsembleOutput has debate_transcript=None by default.

DebateTranscript is imported from debate.py (not inline). The field is Optional with
None default so all existing Phase 9/10 code paths remain unchanged.
"""
from hifi.collective.schemas import EnsembleOutput

# Verify the field exists with None default (no construction needed)
fields = EnsembleOutput.model_fields
assert "debate_transcript" in fields, "debate_transcript field missing from EnsembleOutput"
dt_field = fields["debate_transcript"]

print("EnsembleOutput.debate_transcript field:")
print(f"  annotation : {dt_field.annotation}")
print(f"  required   : {dt_field.is_required()}")
print(f"  default    : {dt_field.default}")
print()
assert not dt_field.is_required(), "debate_transcript must default to None"
print("Backward compatibility: CONFIRMED")
print("All existing code that omits debate_transcript gets None by default.")
print()

# Verify the transcript we built in the previous cell round-trips cleanly
as_json = transcript.model_dump_json(indent=2)
restored = DebateTranscript.model_validate_json(as_json)
assert restored.vote_delta == transcript.vote_delta
assert restored.n_agents_changed_vote == transcript.n_agents_changed_vote
assert restored.majority_decision == transcript.majority_decision
print(f"DebateTranscript JSON roundtrip: OK ({len(as_json):,} chars)")
print(f"  vote_delta             : {restored.vote_delta}")
print(f"  majority_decision      : {restored.majority_decision}")
print(f"  n_agents_changed_vote  : {restored.n_agents_changed_vote}")
print(f"  debate_skipped         : {restored.debate_skipped}")


---
## 6  The 2x2 Factorial — Experimental Design (DJ-067)

### 6.1  Why 4 conditions, not 2

The simplest experiment would compare base vs fine-tuned agents. But this conflates
two effects: individual specialisation (fine-tuning) and collective process design
(debate). The 2x2 factorial disentangles them:

$$\text{Interaction} = (D - B) - (C - A)$$

If the interaction is positive: fine-tuned agents benefit *more* from debate than
base agents. This would mean that heterogeneous specialisation (what fine-tuning
creates) is a prerequisite for productive debate — exactly the literature prediction
(Surowiecki, 2004: diversity is necessary for crowd wisdom).

If the interaction is negative: debate erases the fine-tuning advantage
(herding towards the majority suppresses specialised dissent).

### 6.2  Scale: 10 dates × 3 tickers × 4 conditions = 120 runs

10 dates are chosen to span different market regimes (bull/bear/sideways).
3 tickers (AAPL/JPM/XOM) provide cross-sector coverage.
4 conditions give the 2x2 design.
120 runs is the minimum for a publishable factorial analysis with a plausible
effect size (see Cohen, 1992 for power analysis at N=30 per cell).

### 6.3  Debate participation rate

OQ-D03 asks: what fraction of (date, ticker) pairs produce a non-unanimous initial
vote? If the answer is < 20%, debate adds overhead on 80% of runs for no benefit.
A high participation rate (> 50%) is necessary to make debate an economically
worthwhile mechanism.

The Phase 11 evaluation showed unanimous votes on 2023-03-31 for all 3 tickers.
A single date is not representative — hence the multi-date design.


In [ ]:
"""Visualise the 2x2 factorial design and explain the interaction effect."""
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ---- Left: 2x2 grid diagram ----
ax = axes[0]
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

cell_info = [
    (0, 1, 'A', 'Base model\nNo debate',  'lightgray'),
    (1, 1, 'B', 'Base model\nWith debate', 'cornflowerblue'),
    (0, 0, 'C', 'Fine-tuned\nNo debate',  'lightgreen'),
    (1, 0, 'D', 'Fine-tuned\nWith debate', 'seagreen'),
]
for cx, cy, label, desc, color in cell_info:
    rect = plt.Rectangle((cx, cy), 1, 1, facecolor=color, edgecolor='black', lw=1.5, alpha=0.7)
    ax.add_patch(rect)
    ax.text(cx + 0.5, cy + 0.65, label, ha='center', va='center', fontsize=22, fontweight='bold',
            color='white' if color in ('seagreen', 'cornflowerblue') else 'black')
    ax.text(cx + 0.5, cy + 0.30, desc, ha='center', va='center', fontsize=8,
            color='white' if color in ('seagreen', 'cornflowerblue') else 'black')

ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['No debate', 'With debate'], fontsize=10)
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['Fine-tuned', 'Base model'], fontsize=10)
ax.set_title('Figure 6a: 2x2 Factorial Design (DJ-067)\n10 dates x 3 tickers x 4 conditions = 120 runs')
ax.tick_params(left=False, bottom=False)

# Interaction arrows
ax.annotate('Main effect\ndebate: B-A', xy=(1.5, 1.5), xytext=(0.5, 1.5),
            arrowprops={'arrowstyle': '->', 'color': 'steelblue', 'lw': 1.5},
            ha='center', va='center', fontsize=8, color='steelblue')
ax.annotate('Main effect\nFT: C-A', xy=(0.5, 0.5), xytext=(0.5, 1.5),
            arrowprops={'arrowstyle': '->', 'color': 'seagreen', 'lw': 1.5},
            ha='center', va='center', fontsize=8, color='seagreen')

# ---- Right: interaction effect schematic ----
ax2 = axes[1]
x_vals = [0, 1]  # no debate, with debate
# Scenario 1: positive interaction (FT amplifies debate benefit)
base_line = [0.60, 0.63]       # A, B
ft_synergy = [0.70, 0.80]      # C, D  (big jump)
ft_herding = [0.70, 0.68]      # C, D  (herding: debate hurts FT agents)

ax2.plot(x_vals, base_line,  'o-', color='steelblue', lw=2, ms=8, label='Base (A, B)')
ax2.plot(x_vals, ft_synergy, 's-', color='seagreen',  lw=2, ms=8, label='FT synergy (C, D) — positive interaction')
ax2.plot(x_vals, ft_herding, 's--', color='firebrick', lw=2, ms=8, label='FT herding (C, D) — negative interaction')

ax2.set_xticks([0, 1])
ax2.set_xticklabels(['No debate', 'With debate'])
ax2.set_ylabel('Collective accuracy (illustrative)')
ax2.set_ylim(0.50, 0.90)
ax2.set_title('Figure 6b: Interaction effect schematic\nPositive = synergy; Negative = herding')
ax2.legend(fontsize=8)

# Annotate interaction
ax2.annotate('Positive\ninteraction', xy=(1, 0.80), fontsize=8, color='seagreen',
             xytext=(0.6, 0.84), arrowprops={'arrowstyle': '->', 'color': 'seagreen', 'lw': 1.2})
ax2.annotate('Negative\ninteraction', xy=(1, 0.68), fontsize=8, color='firebrick',
             xytext=(0.6, 0.58), arrowprops={'arrowstyle': '->', 'color': 'firebrick', 'lw': 1.2})

plt.suptitle('Figure 6: 2x2 Factorial Experimental Design (DJ-067)', fontsize=11)
plt.tight_layout()
plt.show()

print('Interaction effect formula:  (D - B) - (C - A)')
print('Positive: FT + debate > sum of parts (synergy)')
print('Negative: debate erases FT diversity advantage (herding)')


---
## 7  Wave 1 Summary and Wave 2 Roadmap

### 7.1  What Wave 1 established

Wave 1 delivered four concrete software components — all tested (1071 tests, 0 lint errors)
and documented. None of these components required LLM calls, server infrastructure,
or network access.

**E0-T1 (compliance fix):** The root cause of `technical_v1` GR collapse (0.19% ratio)
is fixed. 200 compliance examples are available. Re-training (E0-T2) can start
immediately with hardware access.

**E1-T1/T2 (FinancialGraph):** The Phase 12 knowledge graph is built, saved, and
loadable. `expand_query_tickers()` produces the expected 2-hop expansion for all
evaluation tickers. The graph is the input to GraphRetriever (Wave 2).

**E3-T1 (debate schemas):** The full Oxford debate data model is in place.
`identify_minority()` classifies agents correctly for all vote patterns.
`compute_vote_delta()` measures herding direction. `EnsembleOutput` is backward compatible.

### 7.2  Wave 2 plan

| Ticket | Deliverable | Dependency |
|---|---|---|
| E0-T2 | Re-train `technical_v1` @ 500 iters with v2 compliance | Hardware |
| E1-T3 | `GraphRetriever` class (LanceDB + graph pre-filter) | E1-T1/T2 done |
| E3-T2 | `run_debate_round()` orchestration function | E3-T1 done |
| E3-T3 | `run_debate_ensemble()` in `ensemble_runner.py` | E3-T2 done |

Wave 2 tickets E1-T3 and E3-T2 are independent of E0-T2 and can proceed in parallel.

### 7.3  Critical path to Phase 12 completion

```
Wave 1 (complete)
  |
  +-- E0-T2 (retrain) -----> E4 (multi-date 120 runs) --> E5 (baselines)
  |
  +-- E1-T3 (GraphRetriever) -> E2 (GraphRAG eval / OQ-K02)
  |
  +-- E3-T2 (run_debate_round) -> E3-T3 -> E4 -> E5

Critical path: E0-T2 || E3-T2 -> E3-T3 -> E4 -> E5
```


In [ ]:
"""Wave 1 artifact checklist."""
checks = [
    (V2_JSONL,                                            'E0-T1: technical_compliance_v2.jsonl'),
    (ROOT / 'src/hifi/knowledge/graph_store.py',          'E1-T1: src/hifi/knowledge/graph_store.py'),
    (ROOT / 'src/hifi/knowledge/graph_construction.py',   'E1-T2: src/hifi/knowledge/graph_construction.py'),
    (ROOT / 'src/hifi/collective/debate.py',              'E3-T1: src/hifi/collective/debate.py'),
    (ROOT / 'tests/unit/test_compliance_examples_v2.py',  'Tests: test_compliance_examples_v2.py'),
    (ROOT / 'tests/unit/test_graph_store.py',             'Tests: test_graph_store.py (30 tests)'),
    (ROOT / 'tests/unit/test_graph_construction.py',      'Tests: test_graph_construction.py (14 tests)'),
    (ROOT / 'tests/unit/test_debate_schemas.py',          'Tests: test_debate_schemas.py (23 tests)'),
    (KG_JSON,                                             'Runtime: data/knowledge_graph/financial_graph.json'),
]

print('Phase 12 Wave 1 — Artifact Checklist')
print('=' * 55)
all_ok = True
for path, label in checks:
    ok = path.exists()
    if not ok:
        all_ok = False
    status = '[OK]     ' if ok else '[ ] MISS '
    print(f'  {status} {label}')

print()
print('Open Questions to resolve in Wave 2-5:')
oqs = [
    ('OQ-K02', 'GraphRAG P@k >= 5% improvement vs flat RAG?', 'E2'),
    ('OQ-M02', 'Multi-date diversity preserved after fine-tuning?',  'E4'),
    ('OQ-D01', 'Does debate increase herding (kappa)?', 'E4'),
    ('OQ-D02', 'Is the 2x2 interaction effect positive?', 'E4'),
    ('OQ-D03', 'What is the debate participation rate?', 'E4'),
]
print()
print(f'{"ID":>8} {"Wave":>6}  Question')
print('-' * 65)
for qid, q, epic in oqs:
    print(f'{qid:>8} {epic:>6}  {q}')

print()
if all_ok:
    print('All Wave 1 artifacts present. Ready for Wave 2.')
else:
    missing = [l for p, l in checks if not p.exists()]
    print(f'{len(missing)} artifact(s) missing:')
    for m in missing:
        print(f'  - {m}')
    if any('v2.jsonl' in m for m in missing):
        print()
        print('To generate the JSONL:')
        print('  uv run python scripts/generate_compliance_examples.py')
    if any('financial_graph' in m for m in missing):
        print()
        print('To generate the knowledge graph JSON: run Section 3 cells above.')
